(sec-9)=
# 9. Discusión crítica y conclusiones

Esta sección responde las preguntas de la sección 6.4 del enunciado, cada una con la evidencia estadística de las secciones 4 a 8, declara las amenazas a la validez y cierra con las conclusiones y el trabajo futuro. Las cifras salen de las celdas anteriores, de modo que cambian con ellas si se vuelve a ejecutar el cuaderno.

(sec-9-1)=
## 9.1 Discusión crítica

In [75]:
# Insumos de la discusión, todos calculados en las secciones anteriores
f1p_m, f1v_m = F1P.mean(), F1V.mean()
mejor_dl_prueba = max(PROFUNDOS, key=lambda m: f1p_m[m])
comunes_atr = sorted(set.intersection(*[set(f["Tres atributos más importantes"].split(", ")) for _, f in INTERP.iterrows()]))
abl_log = ABLACION[ABLACION["Variante"].str.contains("log")].iloc[0]
abl_eq = [ARTICULO_ABL.get(v_.replace("sin ", "", 1), (v_, v_))[0] for v_ in ABLACION.loc[ABLACION["p TOST"] < ALFA, "Variante"]]
dominio_bw = dominante.index[dominante["Categoría"] == "Confusión Blackhole–Wormhole"].tolist()
bp_max = BRECHAS.set_index("Modelo").loc[PROFUNDOS, "Brecha g"]
ece_min = cal_m["ECE"].idxmin()
ece_min_dl = cal_p["ECE"].idxmin()
n_sobre = int((pend_a < 1).sum())
rob = ROBUSTEZ["Índice medio"]
rank_rob = {m: int((rob > rob[m]).sum()) + 1 for m in rob.index}
rank_f1 = {m: int((f1p_m[[x for x in ORDEN if x != "Trivial"]] > f1p_m[m]).sum()) + 1 for m in ORDEN if m != "Trivial"}
n_modelos = len(ORDEN) - 1
dec_rq1 = RQ1["Decisión"].value_counts()
pct_bloque, pct_modelo = 100 * var_bloque / total_var, 100 * var_modelo / total_var
ic_b2 = r2["IC 95 % por bloques"]
techo_cl = TECHO.iloc[0]
min_lgbm, min_dl = costo.loc["LightGBM", "Minutos"], costo.loc[MEJOR_DL, "Minutos"]

interpretar(f'''**¿Qué modelo funciona mejor y por qué?** Ninguno de forma inequívoca. En prueba, el F1 macro más alto es el de LightGBM ({num(f1p_m["LightGBM"], 4)}), y entre los profundos el de {mejor_dl_prueba} ({num(f1p_m[mejor_dl_prueba], 4)}); por validación, el mejor profundo es {MEJOR_DL} ({num(f1v_m[MEJOR_DL], 4)}). Sin embargo, el *Model Confidence Set* al 90 % conserva {len(MCS)} de los {n_modelos} modelos no triviales, de modo que con {NB_PRU} series de prueba los datos no separan un ganador. En cuanto al porqué, la ablación sobre {MEJOR_DL} muestra que el componente del pipeline con más evidencia es la transformación log(1 + x), cuya ausencia resta {num(abs(abl_log["Diferencia con el completo"]), 4)} de F1 en prueba y {num(abs(abl_log["Diferencia en validación"]), 4)} en validación{f", mientras que {enumerar(abl_eq)} resultan equivalentes a no usarlos dentro de ±δ" if abl_eq else ""}; así mismo, las atribuciones de los tres mejores coinciden en {enumerar(comunes_atr)}, atributos de pérdida y tamaño de paquetes. Es decir, lo que más pesa es la representación de los atributos, que todos los modelos comparten, y no la arquitectura.

**¿Qué modelo generaliza y se calibra mejor?** Ningún modelo sobreajusta el entrenamiento: la brecha entre entrenamiento y validación de los profundos no pasa de {num(bp_max.max(), 4)} ({bp_max.idxmax()}), por debajo de δg = 0,05. Lo que separa a los modelos es el paso a series nuevas: los profundos pierden en mediana {num(caida_dl, 4)} de F1 entre validación y prueba, la regresión logística {num(caida_log, 4)} y LightGBM {num(caida_lgbm, 4)}, de modo que el árbol potenciado es el que mejor generaliza a las series de prueba. En calibración, el menor ECE de prueba es el de {ece_min} ({num(cal_m.loc[ece_min, "ECE"], 4)}), y entre los profundos, el de {ece_min_dl} ({num(cal_p.loc[ece_min_dl, "ECE"], 4)}); {n_sobre} de los {len(pend_a)} modelos tienen pendiente de calibración menor que 1 en prueba, es decir, son sobreconfiados en series nuevas, aunque en validación su temperatura óptima queda cerca de 1.

**¿Qué modelo es más robusto ante perturbaciones y cambio de distribución?** Ante perturbaciones, el más robusto es {mas_rob} (índice {num(rob[mas_rob], 3)}) y el menos, {menos_rob} ({num(rob[menos_rob], 3)}); LightGBM queda en el puesto {rank_rob["LightGBM"]} de {len(rob)}, y la correlación de Spearman entre el F1 limpio y la robustez es de {num(rho_rob.statistic, 2)} ({valor_p(rho_rob.pvalue)}), de modo que robustez y desempeño son ejes distintos. Ante el cambio entre series, en cambio, el más robusto es LightGBM, por su caída menor entre validación y prueba. Esto plantea un compromiso: el modelo que mejor generaliza a series nuevas es de los más frágiles ante ruido y atributos faltantes, y {mas_rob}, el más robusto ante perturbaciones, ocupa el puesto {rank_f1[mas_rob]} de {n_modelos} en F1 de prueba.

**¿Son las diferencias estadísticamente significativas y prácticamente relevantes?** Solo algunas. En RQ1, ningún modelo del Top 10 supera al baseline del Entregable 1 por más de δ: de los 10 contrastes, {enumerar([f"{v} {('es ' if v == 1 else 'son ') if i == 0 else ''}{DECISION_TXT.get(k, (k.lower(), k.lower()))[0 if v == 1 else 1]}" for i, (k, v) in enumerate(dec_rq1.items())])}. En RQ2, LightGBM supera a {MEJOR_DL} por {num(abs(r2["Diferencia media"]), 4)}, una diferencia significativa y relevante según la regla preregistrada, pero que desaparece en la validación cruzada 5 × 2 ({num(d_cv.mean(), 4)}; {valor_p(nb["p"])}) y cuyo IC por bloques, {intervalo(ic_b2)}, contiene al cero: es real en estas series, pero no se puede afirmar que se repita en otras. En RQ3, los modelos que leen los atributos como secuencia y los que no suponen orden {"son equivalentes" if RQ3_DECISION == "Equivalente" else "quedan como «" + RQ3_DECISION.lower() + "»"}, con una diferencia de {num(coef, 4)} en el modelo mixto. La descomposición de la varianza lo resume: el bloque de prueba explica el {num(pct_bloque, 1)} % de la varianza del F1 por bloque, y el modelo, el {num(pct_modelo, 1)} %.

**¿Existe un compromiso relevante entre desempeño y costo?** No en la dirección esperada. Entre los profundos, la correlación entre parámetros y F1 es de {num(rho_p.statistic, 2)} ({valor_p(rho_p.pvalue)}), y LightGBM, que tiene el mayor F1 de prueba, entrena en {num(min_lgbm, 2)} minutos por corrida, frente a {num(min_dl, 2)} de {MEJOR_DL}. Además, en RQ5 {f"{enumerar(equiv5)} {'es equivalente' if len(equiv5) == 1 else 'son equivalentes'} a {MEJOR_DL} dentro de ±δ con menor latencia" if equiv5 else f"ningún modelo más barato es equivalente a {MEJOR_DL} dentro de ±δ"}, de modo que pagar más cómputo no compra F1 en este problema.

**¿Qué tan cerca está el mejor modelo del techo?** Frente al aprendizaje confiado, el techo de exactitud es {num(techo_cl["Exactitud máxima alcanzable"], 4)}, y {mejor_t} llega a {num(acc[mejor_t], 4)}, con γ = {num(g_cl, 3)}; frente a la cota de Cover y Hart, γ = {num(g_ch, 3)}. {"Queda, por tanto, un margen real de mejora" if min(g_cl, g_ch) >= 0.5 else "La mayor parte de la distancia entre el baseline y el techo ya está cubierta"}, que, por lo que muestran los errores (sección 4.5), está concentrado en la frontera entre Blackhole y Wormhole.

**¿Se confirman o se rechazan las hipótesis?** En RQ1 no se rechaza H0 a favor del Top 10: ningún modelo supera al baseline por más de δ. En RQ2 se rechaza H0, en contra del modelo profundo, aunque la ventaja de LightGBM queda acotada a las series de esta prueba. En RQ3 no se rechaza H0, y TOST respalda la equivalencia de los dos grupos. En RQ4, exploratoria, la limitación del paper guía {"es compartida" if compartida else "no es compartida"} por el Top 10: con la partición agrupada, la LSTM confunde Blackhole y Wormhole en el {num(100 * lstm_bench, 2)} % de esos flujos, y los diez modelos, entre el {num(100 * top10_bench.min(), 2)} % y el {num(100 * top10_bench.max(), 2)} %. En RQ5, exploratoria, la respuesta es {"afirmativa" if equiv5 else "negativa"}.''')

**¿Qué modelo funciona mejor y por qué?** Ninguno de forma inequívoca. En prueba, el F1 macro más alto es el de LightGBM (0,8748), y entre los profundos el de CNN-LSTM (0,8626); por validación, el mejor profundo es CNN 1D (0,9299). Sin embargo, el *Model Confidence Set* al 90 % conserva 10 de los 15 modelos no triviales, de modo que con 24 series de prueba los datos no separan un ganador. En cuanto al porqué, la ablación sobre CNN 1D muestra que el componente del pipeline con más evidencia es la transformación log(1 + x), cuya ausencia resta 0,0127 de F1 en prueba y 0,0416 en validación, mientras que el *dropout* y la ponderación de clases resultan equivalentes a no usarlos dentro de ±δ; así mismo, las atribuciones de los tres mejores coinciden en MeanPacketSize y PacketDropRate, atributos de pérdida y tamaño de paquetes. Es decir, lo que más pesa es la representación de los atributos, que todos los modelos comparten, y no la arquitectura.

**¿Qué modelo generaliza y se calibra mejor?** Ningún modelo sobreajusta el entrenamiento: la brecha entre entrenamiento y validación de los profundos no pasa de 0,0154 (BiLSTM), por debajo de δg = 0,05. Lo que separa a los modelos es el paso a series nuevas: los profundos pierden en mediana 0,0715 de F1 entre validación y prueba, la regresión logística 0,0937 y LightGBM 0,0123, de modo que el árbol potenciado es el que mejor generaliza a las series de prueba. En calibración, el menor ECE de prueba es el de LightGBM (0,0356), y entre los profundos, el de CNN-LSTM (0,0467); 14 de los 15 modelos tienen pendiente de calibración menor que 1 en prueba, es decir, son sobreconfiados en series nuevas, aunque en validación su temperatura óptima queda cerca de 1.

**¿Qué modelo es más robusto ante perturbaciones y cambio de distribución?** Ante perturbaciones, el más robusto es LSTM (índice 0,840) y el menos, Logística (0,409); LightGBM queda en el puesto 13 de 15, y la correlación de Spearman entre el F1 limpio y la robustez es de -0,31 (p = 0,254), de modo que robustez y desempeño son ejes distintos. Ante el cambio entre series, en cambio, el más robusto es LightGBM, por su caída menor entre validación y prueba. Esto plantea un compromiso: el modelo que mejor generaliza a series nuevas es de los más frágiles ante ruido y atributos faltantes, y LSTM, el más robusto ante perturbaciones, ocupa el puesto 13 de 15 en F1 de prueba.

**¿Son las diferencias estadísticamente significativas y prácticamente relevantes?** Solo algunas. En RQ1, ningún modelo del Top 10 supera al baseline del Entregable 1 por más de δ: de los 10 contrastes, 3 son no concluyentes, 3 equivalentes, 3 significativos y relevantes y 1 significativo sin relevancia. En RQ2, LightGBM supera a CNN 1D por 0,0164, una diferencia significativa y relevante según la regla preregistrada, pero que desaparece en la validación cruzada 5 × 2 (-0,0003; p = 0,990) y cuyo IC por bloques, [-0,0553; 0,0257], contiene al cero: es real en estas series, pero no se puede afirmar que se repita en otras. En RQ3, los modelos que leen los atributos como secuencia y los que no suponen orden son equivalentes, con una diferencia de -0,0030 en el modelo mixto. La descomposición de la varianza lo resume: el bloque de prueba explica el 93,1 % de la varianza del F1 por bloque, y el modelo, el 0,3 %.

**¿Existe un compromiso relevante entre desempeño y costo?** No en la dirección esperada. Entre los profundos, la correlación entre parámetros y F1 es de 0,07 (p = 0,817), y LightGBM, que tiene el mayor F1 de prueba, entrena en 0,24 minutos por corrida, frente a 1,97 de CNN 1D. Además, en RQ5 Autoencoder, DBN, DNN y Transformer son equivalentes a CNN 1D dentro de ±δ con menor latencia, de modo que pagar más cómputo no compra F1 en este problema.

**¿Qué tan cerca está el mejor modelo del techo?** Frente al aprendizaje confiado, el techo de exactitud es 0,9246, y LightGBM llega a 0,8847, con γ = 0,665; frente a la cota de Cover y Hart, γ = 0,711. Queda, por tanto, un margen real de mejora, que, por lo que muestran los errores (sección {ref}`4.5 <sec-4-5>`), está concentrado en la frontera entre Blackhole y Wormhole.

**¿Se confirman o se rechazan las hipótesis?** En RQ1 no se rechaza H0 a favor del Top 10: ningún modelo supera al baseline por más de δ. En RQ2 se rechaza H0, en contra del modelo profundo, aunque la ventaja de LightGBM queda acotada a las series de esta prueba. En RQ3 no se rechaza H0, y TOST respalda la equivalencia de los dos grupos. En RQ4, exploratoria, la limitación del paper guía es compartida por el Top 10: con la partición agrupada, la LSTM confunde Blackhole y Wormhole en el 15,69 % de esos flujos, y los diez modelos, entre el 12,30 % y el 18,51 %. En RQ5, exploratoria, la respuesta es afirmativa.

(sec-9-2)=
## 9.2 Amenazas a la validez y limitaciones

In [76]:
# Número de caminos examinados (sección 9.3 del enunciado), contados sobre los registros de las secciones anteriores
n_cfg_ajuste, n_arq_f1, n_finalistas, n_corridas = len(ajuste), len(f1), int(FASE2["Finalistas"].sum()), len(registro_exp)
n_confirmatorios = len(RQ1) + len(RQ2) + 1
n_p_ab = len(NA) * (3 * K + 1) + int(NA["p permutación"].notna().sum()) + 2 * len(TABLA_B)
interpretar(f'''**Validez interna.** El mejor modelo profundo y los hiperparámetros se eligen con {NB_VAL} bloques de validación, y la fase 1 de la búsqueda se entrenó con el 25 % de los bloques (desviación 1), lo que hizo que su orden informara poco el de la fase 2 (ρ de {num(FASE2["ρ de Spearman fase 1–fase 2"].min(), 2)} a {num(FASE2["ρ de Spearman fase 1–fase 2"].max(), 2)}); por eso, la conclusión sobre los cuatro modelos con búsqueda se limita a su mejor finalista. La suspensión del equipo durante la fase 2 solo afectó el tiempo de seis corridas, que se corrigió. La réplica del paper guía valida el pipeline con cuatro de sus cinco modelos, pero no reproduce el resultado clave de la LSTM.

**Validez externa.** Todo el benchmark usa una sola versión de un solo conjunto de datos, con {NB_PRU} series de prueba aproximadas por bloques de 1000 flujos, y la serie explica el {num(pct_bloque, 1)} % de la varianza del F1; con otra elección de series, el orden de los modelos puede cambiar, como sugiere la validación cruzada 5 × 2 de RQ2. Además, el atajo del puerto 654 (sección 4.5) indica que parte del desempeño depende de la configuración de los escenarios simulados.

**Validez de constructo.** El F1 macro pesa por igual a las cinco clases, aunque en la práctica no todos los errores cuestan lo mismo: confundir dos ataques de enrutamiento es menos grave que dejar pasar un ataque como tráfico normal. Así mismo, la latencia y la energía se midieron en un solo equipo, en CPU y con 2 hilos por corrida, y el F1 por bloque se promedia sobre las clases presentes en cada bloque (aclaración 2 de la adenda).

**Validez estadística.** Con R = {R} semillas, el tope de cómputo, la diferencia mínima detectable en RQ1 es de {num(ADENDA["piloto"]["diferencia_minima_detectable"], 4)}, mayor que δ, de modo que una mejora real de magnitud δ podría pasar inadvertida; por eso los resultados no significativos se leen con TOST y con la prueba bayesiana. Las pruebas por filas, como McNemar y DeLong, son optimistas porque tratan los flujos como independientes, y las preguntas exploratorias RQ4 y RQ5 no tienen control del error por familia.

**Análisis examinados y tasa de falsos descubrimientos.** Para dimensionar los caminos que se bifurcan (sección 9.3 del enunciado), el benchmark examinó {len(ORDEN)} modelos, {n_cfg_ajuste} configuraciones de hiperparámetros en el ajuste, {num(n_arq_f1, 0)} arquitecturas en la fase 1 y {n_finalistas} finalistas en la fase 2, con {num(n_corridas, 0)} corridas registradas en total y {len(NOMBRES_M)} métricas por corrida. Las afirmaciones confirmatorias se apoyan solo en los {n_confirmatorios} contrastes preregistrados de RQ1 a RQ3, con Holm dentro de cada familia, de modo que las demás pruebas no aumentan su tasa de error. En cambio, solo las tablas A y B suman al menos {n_p_ab} valores p sin corrección por multiplicidad, y si todas sus hipótesis nulas fueran ciertas cabría esperar unos {num(0.05 * n_p_ab, 0)} rechazos por azar; por eso se leen como evidencia de dirección y no como hallazgos. Así mismo, elegir el máximo entre tantas configuraciones infla el puntaje de validación: el F1 interno del ajuste supera al de prueba en {num(np.mean(interno - externo), 4)} en promedio (sección 7), que es el sesgo del ganador acumulado junto con el cambio entre series. Por último, el umbral de ECE de la condición (iii) del criterio de adecuación se fijó después de ver los resultados (sección 7), de modo que ese veredicto es exploratorio y no sustenta ninguna conclusión confirmatoria.''')

**Validez interna.** El mejor modelo profundo y los hiperparámetros se eligen con 19 bloques de validación, y la fase 1 de la búsqueda se entrenó con el 25 % de los bloques (desviación 1), lo que hizo que su orden informara poco el de la fase 2 (ρ de -0,42 a 0,02); por eso, la conclusión sobre los cuatro modelos con búsqueda se limita a su mejor finalista. La suspensión del equipo durante la fase 2 solo afectó el tiempo de seis corridas, que se corrigió. La réplica del paper guía valida el pipeline con cuatro de sus cinco modelos, pero no reproduce el resultado clave de la LSTM.

**Validez externa.** Todo el benchmark usa una sola versión de un solo conjunto de datos, con 24 series de prueba aproximadas por bloques de 1000 flujos, y la serie explica el 93,1 % de la varianza del F1; con otra elección de series, el orden de los modelos puede cambiar, como sugiere la validación cruzada 5 × 2 de RQ2. Además, el atajo del puerto 654 (sección {ref}`4.5 <sec-4-5>`) indica que parte del desempeño depende de la configuración de los escenarios simulados.

**Validez de constructo.** El F1 macro pesa por igual a las cinco clases, aunque en la práctica no todos los errores cuestan lo mismo: confundir dos ataques de enrutamiento es menos grave que dejar pasar un ataque como tráfico normal. Así mismo, la latencia y la energía se midieron en un solo equipo, en CPU y con 2 hilos por corrida, y el F1 por bloque se promedia sobre las clases presentes en cada bloque (aclaración 2 de la adenda).

**Validez estadística.** Con R = 20 semillas, el tope de cómputo, la diferencia mínima detectable en RQ1 es de 0,0162, mayor que δ, de modo que una mejora real de magnitud δ podría pasar inadvertida; por eso los resultados no significativos se leen con TOST y con la prueba bayesiana. Las pruebas por filas, como McNemar y DeLong, son optimistas porque tratan los flujos como independientes, y las preguntas exploratorias RQ4 y RQ5 no tienen control del error por familia.

**Análisis examinados y tasa de falsos descubrimientos.** Para dimensionar los caminos que se bifurcan (sección 9.3 del enunciado), el benchmark examinó 16 modelos, 180 configuraciones de hiperparámetros en el ajuste, 4 000 arquitecturas en la fase 1 y 84 finalistas en la fase 2, con 6 471 corridas registradas en total y 14 métricas por corrida. Las afirmaciones confirmatorias se apoyan solo en los 12 contrastes preregistrados de RQ1 a RQ3, con Holm dentro de cada familia, de modo que las demás pruebas no aumentan su tasa de error. En cambio, solo las tablas A y B suman al menos 276 valores p sin corrección por multiplicidad, y si todas sus hipótesis nulas fueran ciertas cabría esperar unos 14 rechazos por azar; por eso se leen como evidencia de dirección y no como hallazgos. Así mismo, elegir el máximo entre tantas configuraciones infla el puntaje de validación: el F1 interno del ajuste supera al de prueba en 0,0796 en promedio (sección 7), que es el sesgo del ganador acumulado junto con el cambio entre series. Por último, el umbral de ECE de la condición (iii) del criterio de adecuación se fijó después de ver los resultados (sección 7), de modo que ese veredicto es exploratorio y no sustenta ninguna conclusión confirmatoria.

(sec-9-3)=
## 9.3 Conclusiones y trabajo futuro

In [77]:
interpretar(f'''**Conclusiones.**

1. En UAVIDS-2025, con una partición agrupada por series y el mismo protocolo para todos, ningún modelo profundo del Top 10 supera al baseline del Entregable 1, {base}, por más de δ = 0,01 (RQ1): {enumerar(RQ1.loc[RQ1["Decisión"] == "Equivalente", "a"].tolist()) if (RQ1["Decisión"] == "Equivalente").any() else "ninguno"} {"es equivalente" if (RQ1["Decisión"] == "Equivalente").sum() == 1 else "son equivalentes"} a él dentro de ±δ, y ninguno alcanza el umbral de mejora de 0,8775 que fijó el Entregable 1.
2. LightGBM obtiene el mayor F1 macro de prueba, {num(f1p_m["LightGBM"], 4)}, y supera al mejor modelo profundo por validación, {MEJOR_DL}, de forma significativa y relevante (RQ2); sin embargo, la ventaja no se repite en la validación cruzada 5 × 2, de modo que depende de las series de esta prueba y no basta para descartar a los modelos profundos.
3. La serie pesa más que la arquitectura: el bloque de prueba explica el {num(pct_bloque, 1)} % de la varianza del F1 por bloque, frente al {num(pct_modelo, 1)} % del modelo, y los modelos secuenciales y sin orden son equivalentes (RQ3). En consecuencia, la ganancia del Entregable 3 no vendrá de una arquitectura más compleja.
4. La confusión entre Blackhole y Wormhole es la limitación compartida por las diez familias del Top 10 (RQ4) {f"y la categoría que más errores reúne en {enumerar(dominio_bw)}" if dominio_bw else ""}, de modo que es el hueco de investigación que se traslada al Entregable 3; el resultado clave del paper guía, el 94 % de la LSTM, no se reproduce con su propio protocolo.
5. El pipeline es reproducible: en un entorno limpio, las corridas repetidas dan el mismo F1 hasta el último decimal, y la tabla principal se recalcula desde las corridas guardadas.

**Trabajo futuro.** Para el Entregable 3 se propone un modelo original que extienda el paper guía con un mecanismo dirigido a la frontera entre Blackhole y Wormhole, como la pérdida contrastiva supervisada o la mezcla de negativos difíciles, o una etapa jerárquica dedicada a ese par, y que se compare con {base} y con LightGBM con el mismo protocolo. Así mismo, conviene elegir el modelo con validación cruzada por bloques, en lugar de una sola partición de validación, para reducir la dependencia de la serie que mostró RQ2; recalibrar las probabilidades con datos de la serie de destino; exigir robustez ante ruido y atributos faltantes, y, cuando exista, evaluar el modelo en un segundo conjunto de datos de intrusiones en redes de drones.''')

**Conclusiones.**

1. En UAVIDS-2025, con una partición agrupada por series y el mismo protocolo para todos, ningún modelo profundo del Top 10 supera al baseline del Entregable 1, CNN 1D, por más de δ = 0,01 (RQ1): Autoencoder, Transformer y DBN son equivalentes a él dentro de ±δ, y ninguno alcanza el umbral de mejora de 0,8775 que fijó el Entregable 1.
2. LightGBM obtiene el mayor F1 macro de prueba, 0,8748, y supera al mejor modelo profundo por validación, CNN 1D, de forma significativa y relevante (RQ2); sin embargo, la ventaja no se repite en la validación cruzada 5 × 2, de modo que depende de las series de esta prueba y no basta para descartar a los modelos profundos.
3. La serie pesa más que la arquitectura: el bloque de prueba explica el 93,1 % de la varianza del F1 por bloque, frente al 0,3 % del modelo, y los modelos secuenciales y sin orden son equivalentes (RQ3). En consecuencia, la ganancia del Entregable 3 no vendrá de una arquitectura más compleja.
4. La confusión entre Blackhole y Wormhole es la limitación compartida por las diez familias del Top 10 (RQ4) y la categoría que más errores reúne en CNN 1D y DBN, de modo que es el hueco de investigación que se traslada al Entregable 3; el resultado clave del paper guía, el 94 % de la LSTM, no se reproduce con su propio protocolo.
5. El pipeline es reproducible: en un entorno limpio, las corridas repetidas dan el mismo F1 hasta el último decimal, y la tabla principal se recalcula desde las corridas guardadas.

**Trabajo futuro.** Para el Entregable 3 se propone un modelo original que extienda el paper guía con un mecanismo dirigido a la frontera entre Blackhole y Wormhole, como la pérdida contrastiva supervisada o la mezcla de negativos difíciles, o una etapa jerárquica dedicada a ese par, y que se compare con CNN 1D y con LightGBM con el mismo protocolo. Así mismo, conviene elegir el modelo con validación cruzada por bloques, en lugar de una sola partición de validación, para reducir la dependencia de la serie que mostró RQ2; recalibrar las probabilidades con datos de la serie de destino; exigir robustez ante ruido y atributos faltantes, y, cuando exista, evaluar el modelo en un segundo conjunto de datos de intrusiones en redes de drones.